In [0]:
dbutils.library.restartPython()

In [0]:
import sys

sys.path.append("/Workspace/Users/sjaddya98@gmail.com/energy-lakehouse/src")

In [0]:
from pyspark.sql.types import IntegerType, StringType, StructField, StructType

from energy_lakehouse.silver import series_lookup

print("session timezone was:", spark.conf.get("spark.sql.session.timeZone"))
spark.conf.set("spark.sql.session.timeZone", "UTC")

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

SERIES_SCHEMA = StructType(
    [
        StructField("filter_id", IntegerType(), False),
        StructField("series_name", StringType(), False),
        StructField("series_group", StringType(), False),
        StructField("unit", StringType(), False),
    ]
)

spark.createDataFrame(series_lookup(), SERIES_SCHEMA).createOrReplaceTempView("series_lookup")
spark.table("series_lookup").show(n=20, truncate=False)

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.silver.power_hourly AS
SELECT
    b.timestamp_utc                                            AS hour_start_utc,
    from_utc_timestamp(b.timestamp_utc, 'Europe/Berlin')       AS hour_start_berlin,
    date(from_utc_timestamp(b.timestamp_utc, 'Europe/Berlin')) AS date_berlin,
    hour(from_utc_timestamp(b.timestamp_utc, 'Europe/Berlin')) AS hour_of_day_berlin,
    b.filter_id,
    s.series_name,
    s.series_group,
    s.unit,
    b.value,
    b.load_id,
    b.fetched_at
FROM workspace.bronze.smard_raw AS b
LEFT JOIN series_lookup AS s
       ON s.filter_id = b.filter_id

In [0]:
%sql
SELECT
  (SELECT count(*) FROM workspace.bronze.smard_raw)                                 AS bronze_rows,
  (SELECT count(*) FROM workspace.silver.power_hourly)                              AS silver_rows,
  (SELECT count(*) FROM workspace.silver.power_hourly WHERE series_name IS NULL)     AS unnamed_rows,
  (SELECT count(*) FROM (
     SELECT filter_id, hour_start_utc
     FROM workspace.silver.power_hourly
     GROUP BY filter_id, hour_start_utc HAVING count(*) > 1))                        AS duplicate_keys

In [0]:
%sql
SELECT date_berlin, count(DISTINCT hour_start_utc) AS hours
FROM workspace.silver.power_hourly
WHERE series_name = 'day_ahead_price_de_lu'
  AND date_berlin IN (DATE '2026-10-24', DATE '2026-10-25', DATE '2026-10-26')
GROUP BY date_berlin
ORDER BY date_berlin

In [0]:
%sql
SELECT hour_of_day_berlin, count(*) AS times
FROM workspace.silver.power_hourly
WHERE series_name = 'day_ahead_price_de_lu'
  AND date_berlin = DATE '2026-10-25'
GROUP BY hour_of_day_berlin
HAVING count(*) > 1

In [0]:
%sql
SELECT series_group, count(DISTINCT series_name) AS series, count(*) AS rows, min(hour_start_utc) AS first_hour, max(hour_start_utc) AS last_hour
FROM workspace.silver.power_hourly
GROUP BY series_group
ORDER BY series_group

In [0]:
%sql
SELECT series_name,
       count(*)     AS rows,
       count(value) AS realised_rows,
       max(hour_start_utc) AS last_hour,
       max(CASE WHEN value IS NOT NULL THEN hour_start_utc END) AS last_realised_hour
FROM workspace.silver.power_hourly
GROUP BY series_name
ORDER BY series_name

In [0]:
%sql
SELECT date_trunc('MONTH', hour_start_utc) AS month,
       count(DISTINCT hour_start_utc)      AS hours
FROM workspace.silver.power_hourly
WHERE series_name = 'grid_load'
GROUP BY month
ORDER BY month